In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [2]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [3]:
df_train.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001688,Male,Yes,1,Not Graduate,No,3500,1083.0,135.0,360.0,1.0,Urban,Y
1,LP002234,Male,No,0,Graduate,Yes,7167,0.0,128.0,360.0,1.0,Urban,Y
2,LP002237,Male,No,1,Graduate,NaN,3667,0.0,113.0,180.0,1.0,Urban,Y
3,LP002366,Male,Yes,0,Graduate,No,2666,4300.0,121.0,360.0,1.0,Rural,Y
4,LP002893,Male,No,0,Graduate,No,1836,33837.0,90.0,360.0,1.0,Urban,N


In [4]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            266 non-null    object 
 1   Gender             262 non-null    object 
 2   Married            266 non-null    object 
 3   Dependents         258 non-null    object 
 4   Education          266 non-null    object 
 5   Self_Employed      251 non-null    object 
 6   ApplicantIncome    266 non-null    int64  
 7   CoapplicantIncome  266 non-null    float64
 8   LoanAmount         266 non-null    float64
 9   Loan_Amount_Term   258 non-null    float64
 10  Credit_History     240 non-null    float64
 11  Property_Area      266 non-null    object 
 12  Loan_Status        266 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 27.1+ KB


In [5]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            115 non-null    object 
 1   Gender             114 non-null    object 
 2   Married            115 non-null    object 
 3   Dependents         115 non-null    object 
 4   Education          115 non-null    object 
 5   Self_Employed      109 non-null    object 
 6   ApplicantIncome    115 non-null    int64  
 7   CoapplicantIncome  115 non-null    float64
 8   LoanAmount         115 non-null    float64
 9   Loan_Amount_Term   112 non-null    float64
 10  Credit_History     111 non-null    float64
 11  Property_Area      115 non-null    object 
dtypes: float64(4), int64(1), object(7)
memory usage: 10.9+ KB


In [6]:
df_train['Loan_Status'] = df_train['Loan_Status'].map({'Y': 1, 'N': 0})
df_train = df_train.astype({'Credit_History': object})

In [7]:
df_train.drop(columns=['Loan_ID'], inplace=True)
df_test.drop(columns = ['Loan_ID'], inplace = True)

In [8]:
for i in df_train.select_dtypes('object').columns:
    print(df_train[i].value_counts(), '\n')

Gender
Male      213
Female     49
Name: count, dtype: int64 

Married
Yes    160
No     106
Name: count, dtype: int64 

Dependents
0     159
2      42
1      37
3+     20
Name: count, dtype: int64 

Education
Graduate        191
Not Graduate     75
Name: count, dtype: int64 

Self_Employed
No     227
Yes     24
Name: count, dtype: int64 

Credit_History
1.0    198
0.0     42
Name: count, dtype: int64 

Property_Area
Semiurban    99
Urban        88
Rural        79
Name: count, dtype: int64 



In [9]:
df_train.isnull().sum()

Gender                4
Married               0
Dependents            8
Education             0
Self_Employed        15
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            0
Loan_Amount_Term      8
Credit_History       26
Property_Area         0
Loan_Status           0
dtype: int64

In [10]:
categorical_features = df_train.select_dtypes(include=['object']).columns.tolist()
numerical_features = df_train.select_dtypes(exclude=['object']).columns.tolist()

In [11]:
print("Categorical Features:", categorical_features)
print("Numerical Features:", numerical_features)

Categorical Features: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']
Numerical Features: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Loan_Status']


In [12]:
cat_imputer = SimpleImputer(strategy='most_frequent')
num_imputer = SimpleImputer(strategy='median')

In [13]:
df_train[categorical_features] = cat_imputer.fit_transform(df_train[categorical_features])
df_train[numerical_features] = num_imputer.fit_transform(df_train[numerical_features])

In [14]:
df_test['Loan_Status'] = 999

In [15]:
df_test[categorical_features] = pd.DataFrame(
    cat_imputer.transform(df_test[categorical_features]),
    columns=categorical_features,
    index=df_test.index
)

df_test[numerical_features] = pd.DataFrame(
    num_imputer.transform(df_test[numerical_features]),
    columns=numerical_features,
    index=df_test.index
)


In [16]:
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col])
    df_test[col] = le.transform(df_test[col])
    label_encoders[col] = le

In [17]:
X = df_train.drop(columns=['Loan_Status'])
y = df_train['Loan_Status']

In [18]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [19]:
df_test.drop(columns = ['Loan_Status'], inplace = True)

In [20]:
rf = RandomForestClassifier(n_estimators=50, max_depth=3, min_samples_split=2, random_state=42)

In [21]:
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Accuracy = {acc:.4f}, F1 Score = {f1:.4f}")
print(classification_report(y_val, y_pred))

Accuracy = 0.8519, F1 Score = 0.9048
              precision    recall  f1-score   support

         0.0       1.00      0.50      0.67        16
         1.0       0.83      1.00      0.90        38

    accuracy                           0.85        54
   macro avg       0.91      0.75      0.79        54
weighted avg       0.88      0.85      0.83        54



In [22]:
y_train_pred = rf.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

print(f"Train Accuracy = {train_acc:.4f}, Train F1 Score = {train_f1:.4f}")
print(classification_report(y_train, y_train_pred))

Train Accuracy = 0.8491, Train F1 Score = 0.9030
              precision    recall  f1-score   support

         0.0       0.94      0.51      0.66        61
         1.0       0.83      0.99      0.90       151

    accuracy                           0.85       212
   macro avg       0.89      0.75      0.78       212
weighted avg       0.86      0.85      0.83       212



In [ ]:
predictions = rf.predict(df_test)
loan_ids = pd.read_csv('test.csv')['Loan_ID']

df = pd.DataFrame({
    'Loan_ID': loan_ids,
    'Prediction': predictions
})

df.to_csv("submission.csv", index=False)